In [ ]:
# 4_averaging.ipynb
# Cell tag parameters (for batch processing — papermill reads these and loops through the participants)
# PAPERMILL_RUN is toggled "True" if batch processing is being used
# This handles three pipelines: baseline check, the alternative pipeline (without items 2, 61 and 62) and the standard one
p_id = 'participant_1'
PAPERMILL_RUN = False
IS_BASELINE_CHECK = False
ALT_PIPELINE = True

In [ ]:
# Imports
import sys
from IPython.display import display
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import mne

# Allow imports to work regardless of the notebook's location by locating the project root.
# EEG_DIR is defined in config.py
def find_project_root(current_path, marker='config.py'):
    path = Path(current_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path 

root_dir = find_project_root(Path.cwd())
sys.path.append(str(root_dir))

from config import EEG_DIR, ITEMS_TO_DROP, conditions

mne.set_log_level('error') # reduce extraneous MNE output

# Plotting Toggle: it is deactivated if running each notebook manually, but if batch processing
# is being used, it is activated so windows don't pop-up
if PAPERMILL_RUN:
    # Batch Mode: Suppress pop-ups and draw plots inline
    %matplotlib inline
    mne.viz.set_browser_backend('matplotlib')
    show_plots = False
else:
    %matplotlib auto
    show_plots = True

In [ ]:
# Path management
subj_dir = EEG_DIR / p_id

In [ ]:
# Dynamic config
if IS_BASELINE_CHECK:
    save_dir = subj_dir / 'baseline_check'
    load_file = save_dir / f'{p_id}-bc-epo.fif'
    suffix = '-bc-ave.fif'
    tmin, tmax = -1.2, 0.0

elif ALT_PIPELINE:
    # Pipeline without items 2, 61 and 62
    save_dir = subj_dir / 'alternative_pipeline'
    load_file = subj_dir / f'{p_id}-epo.fif'  # Loads standard epochs
    suffix = '_alt-ave.fif'
    tmin, tmax = -0.200, 1.000

else:
    # Normal pipeline
    save_dir = subj_dir
    load_file = save_dir / f'{p_id}-epo.fif'
    suffix = '-ave.fif'
    tmin, tmax = -0.200, 1.000

save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load epochs
epochs = mne.read_epochs(load_file, preload=True).set_montage('easycap-M1')

In [ ]:
if ALT_PIPELINE:
    if epochs.metadata is None:
        raise ValueError(
            f"{p_id}: epochs have no metadata. Re-run the segmenting notebook "
            f"(it attaches Item/Condition) before running the ALT pipeline."
        )
    # Drop items (item metadata was added in the last step)
    # ITEMS_TO_DROP is defined in config.py
    epochs.metadata['Item'] = pd.to_numeric(epochs.metadata['Item'], errors='coerce')
    n_before = len(epochs)
    epochs = epochs[~epochs.metadata['Item'].isin(ITEMS_TO_DROP)]
    print(f"⚠️ ALT PIPELINE: Dropped {n_before - len(epochs)} epochs for items {ITEMS_TO_DROP}")

In [ ]:
# Create a separate evoked object for each condition

# Conditions are defined in config.py

evokeds = {c:epochs[c].average() for c in conditions}

evokeds

In [ ]:
# Plot average ERP for each condition

for c in evokeds.keys():
    evokeds[c].plot_joint(title=c, show=show_plots);

In [ ]:
# Examine contrasts between conditions

mne.viz.plot_compare_evokeds(evokeds, picks='Cz', show=show_plots);

In [ ]:
# Plot the ERPs for specific channels

# define the channels we want plots for
channels = ['Fz', 'Cz', 'Pz', 'Oz']

# create a figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# plot each channel in a separate subplot
for idx, chan in enumerate(channels):
    mne.viz.plot_compare_evokeds(evokeds, 
                                picks=chan,
                                ylim={'eeg':(-10, 10)},
                                show_sensors='lower right',
                                legend='upper left',
                                axes=axes.reshape(-1)[idx],
                                show=False
                                );
plt.show()    

In [ ]:
# Topoplots for specific channels

# Specify times to plot at, as (min, max, stepsize)
times = np.arange(tmin, tmax, 0.1)

for condition in evokeds.keys():
    print(condition)
    fig = evokeds[condition].plot_topomap(times=times, average=0.050, vlim=(-8, 8), show=False)
    fig.suptitle(f'{p_id} — {condition}', fontsize=14)
    if show_plots:
        plt.show()
    else:
        display(fig)
        plt.close(fig)

In [ ]:
# Difference Waves

# Contrast 1: Condition B minus Condition A
diff_B_A = mne.combine_evoked(
    [
     evokeds['ConditionB'], # The "Active" condition (+1)
     evokeds['ConditionA']  # The "Baseline" condition (-1)
    ],
    weights=[1, -1]
)

# Contrast 2: Condition C minus Condition A
diff_C_A = mne.combine_evoked(
    [
     evokeds['ConditionC'], # The "Active" condition (+1)
     evokeds['ConditionA']  # The "Baseline" condition (-1)
    ],
    weights=[1, -1]
)

# Contrast 3: Condition C minus Condition B
diff_C_B = mne.combine_evoked(
    [
        evokeds['ConditionC'],
        evokeds['ConditionB']
    ],
    weights=[1,-1]
)

In [ ]:
# Compare the three difference waves

mne.viz.plot_compare_evokeds({
    'Diff (B - A)': diff_B_A,
    'Diff (C - A)': diff_C_A,
    'Diff (C - B)': diff_C_B
}, picks='Cz', show=show_plots)

In [ ]:
# Topoplot of difference waves

diff_dict = {
    "Condition B - Condition A": diff_B_A,
    "Condition C - Condition A": diff_C_A,
    "Condition C - Condition B": diff_C_B
}

for diff_name, diff_evoked in diff_dict.items():
    fig = diff_evoked.plot_topomap(times=times, average=0.050, vlim=(-8, 8), show=False)
    fig.suptitle(f'{p_id} — Diff: {diff_name}', fontsize=14)
    if show_plots:
        plt.show()
    else:
        display(fig)
        plt.close(fig)

In [ ]:
# Add descriptive labels before saving

diff_B_A.comment = "Condition B - Condition A"

diff_C_A.comment = "Condition C - Condition A"

diff_C_B.comment = "Condition C - Condition B"

In [ ]:
# Save difference waves
mne.write_evokeds(save_dir / f'{p_id}_ConditionB-ConditionA{suffix}', diff_B_A, overwrite=True);
mne.write_evokeds(save_dir / f'{p_id}_ConditionC-ConditionA{suffix}', diff_C_A, overwrite=True);
mne.write_evokeds(save_dir / f'{p_id}_ConditionC-ConditionB{suffix}', diff_C_B, overwrite=True);

In [ ]:
# Save averaged conditions

for condition in evokeds: 
    mne.write_evokeds(save_dir / f'{p_id}_{condition}{suffix}', evokeds[condition], overwrite=True)

In [ ]:
print(f"✅ Averaging (and preprocessing) complete for {p_id}. All files and plots saved.")